<a href="https://colab.research.google.com/github/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/advanced_topics/ct_seg_to_tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading CT images and DICOM segmentations into PyTorch tensors

Updated: October 2026

Many collections in [NCI Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov/) contain CT
images together with segmentations stored as [DICOM Segmentation (SEG)](https://highdicom.readthedocs.io/en/latest/seg.html)
objects. These are a great resource for training deep learning segmentation models, but there are a few
steps between "a folder of DICOM files" and "a pair of tensors ready for a model".

In this tutorial we will:

1. Use `idc-index` to find CT series that have a corresponding SEG, and download a few of them.
2. Use `highdicom` to load the CT series as a 3D volume in Hounsfield units (HU).
3. Use `highdicom` to load the SEG as a 3D mask array that lines up voxel-for-voxel with the CT.
4. Convert both to PyTorch tensors and wrap them in a simple `torch.utils.data.Dataset`.

We use the [C4KC-KiTS](https://www.cancerimagingarchive.net/collection/c4kc-kits/) collection, which contains
abdominal CTs with manual segmentations of the kidneys and kidney tumors.

This tutorial assumes basic familiarity with DICOM, `highdicom` and PyTorch. If you are new to IDC, start with
the [Getting Started](https://github.com/ImagingDataCommons/IDC-Tutorials/tree/master/notebooks/getting_started)
notebooks. No cloud account or GPU is needed.

## Setup

Install the packages we need. `idc-index` is used to search and download IDC data, and `highdicom`
(which builds on `pydicom`) is used to read the images and segmentations. If this upgrades a package that was
already installed, restart the kernel before continuing.

In [ ]:
%pip install -q "idc-index>=0.13" "highdicom>=0.28" torch matplotlib

In [ ]:
from pathlib import Path

import highdicom as hd
import matplotlib.pyplot as plt
import numpy as np
import pydicom
import torch
from idc_index import IDCClient

In [ ]:
import idc_index
print(idc_index.__version__)


## 1. Find CT series that have a segmentation

`idc-index` ships a table, `index`, with one row per DICOM series in IDC. It also provides additional tables
that can be fetched on demand. One of these, `seg_index`, has one row per SEG series, and its
`segmented_SeriesInstanceUID` column gives the `SeriesInstanceUID` of the image series that was segmented.
This means we can find CT/SEG pairs with a single SQL query, without opening any DICOM files.

To keep the download small, we pick the three smallest pairs from C4KC-KiTS.

In [ ]:
client = IDCClient()
client.fetch_index("seg_index")

query = """
SELECT
  seg_series.PatientID,
  seg_index.segmented_SeriesInstanceUID AS ct_series_uid,
  seg_index.SeriesInstanceUID AS seg_series_uid,
  ct_series.series_size_MB + seg_series.series_size_MB AS download_size_MB
FROM seg_index
JOIN index AS seg_series ON seg_index.SeriesInstanceUID = seg_series.SeriesInstanceUID
JOIN index AS ct_series ON seg_index.segmented_SeriesInstanceUID = ct_series.SeriesInstanceUID
WHERE seg_series.collection_id = 'c4kc_kits'
ORDER BY download_size_MB
LIMIT 3
"""
pairs = client.sql_query(query)
pairs

Now download both series of each pair. By default `idc-index` organizes files into a
`collection/patient/study/series` folder hierarchy. Here we use the `dirTemplate` argument to put each series
directly into a folder named by its `SeriesInstanceUID`, which makes the files easy to find afterwards.

In [ ]:
download_dir = Path("idc_data")

client.download_from_selection(
    downloadDir=str(download_dir),
    seriesInstanceUID=pairs.ct_series_uid.tolist() + pairs.seg_series_uid.tolist(),
    dirTemplate="%SeriesInstanceUID",
)

## 2. Load a CT series as a 3D volume

A CT series is stored as a set of single-frame DICOM files, one per slice. To turn these into a 3D array, the
slices must be sorted by their position in space (the file names and `InstanceNumber` are not reliable for this),
and the stored pixel values must be converted to Hounsfield units using `RescaleSlope` and `RescaleIntercept`.

`highdicom.get_volume_from_series()` does both of these things for us. It returns a `highdicom.Volume`, which
holds:

- `array`: the voxel values, as a NumPy array with axes `(slices, rows, columns)`
- `affine`: a 4×4 matrix mapping voxel indices to positions in the patient coordinate system (in mm)

We will work with the first pair for now.

In [ ]:
case = pairs.iloc[0]

ct_files = sorted((download_dir / case.ct_series_uid).glob("*.dcm"))
ct_datasets = [pydicom.dcmread(f) for f in ct_files]
ct_volume = hd.get_volume_from_series(ct_datasets, dtype=np.float32)

print("CT shape:", ct_volume.shape)
print("Voxel spacing (mm):", ct_volume.spacing)
print("HU range:", ct_volume.array.min(), "to", ct_volume.array.max())
print("Affine:\n", ct_volume.affine)

## 3. Load the segmentation and align it with the CT

A DICOM SEG is a single multi-frame file. that can contain several *segments* (one per structure).
There are two variants: in BINARY SEGs (like those in C4KC-KiTS), each frame stores a binary mask
for one segment on one slice, and for LABELMAP SEGs, segments are combined together in a single
frame for a given slice. Highdicom's `highdicom.seg.Segmentation` class works with both types and
gives consistent results regardless of how frames are stored, hiding this difference from you.
Let's read the file with `highdicom.seg.segread()` and see which segments it contains.

In [ ]:
seg_file = next((download_dir / case.seg_series_uid).glob("*.dcm"))
seg = hd.seg.segread(seg_file)

for segment in seg.SegmentSequence:
    print(segment.SegmentNumber, segment.SegmentLabel)

Each segment also carries coded descriptions of what it represents (for example
`SegmentedPropertyTypeCodeSequence`). These coded values are more reliable than the free-text `SegmentLabel`
when combining segmentations from different sources.

We now need the masks as an array with the same shape and voxel ordering as the CT. A SEG does not have to store
its frames in the same order as the CT slices, and it may omit frames that are empty. Two `highdicom` methods
handle this for us:

- `seg.get_volume()` assembles the frames into a 3D volume. With `combine_segments=False`, the result has one
  binary channel per segment in a trailing axis: `(slices, rows, columns, segments)`. (Use `combine_segments=True`
  instead if you want a single label map in which each voxel holds a segment number.)
- `.match_geometry(ct_volume)` rearranges the mask volume so that it lies on exactly the same voxel grid as the
  CT. This only pads, flips or permutes the array; it never interpolates, and it raises an error if the two
  volumes cannot be aligned.

We pass `tol=1e-4` (in units of voxels) to `match_geometry()` because the SEG files in this collection store
positions and pixel spacing with fewer decimal places than the CT files, so the two grids differ by a tiny
rounding error.

In [ ]:
mask_volume = seg.get_volume(combine_segments=False).match_geometry(ct_volume, tol=1e-4)

print("CT shape:  ", ct_volume.shape)
print("Mask shape:", mask_volume.shape)

Let's check the alignment by overlaying each segment on the CT slice where the segmentation is largest. We use a
soft-tissue display window (-200 to 300 HU) for viewing; this does not change the underlying values.

In [ ]:
masks = mask_volume.array
slice_index = masks.sum(axis=(1, 2, 3)).argmax()

fig, axes = plt.subplots(1, masks.shape[-1], figsize=(10, 5))
for k, ax in enumerate(axes):
    ax.imshow(ct_volume.array[slice_index], cmap="gray", vmin=-200, vmax=300)
    ax.imshow(np.ma.masked_equal(masks[slice_index, :, :, k], 0), cmap="autumn", alpha=0.5)
    ax.set_title(seg.SegmentSequence[k].SegmentLabel)
    ax.axis("off")
plt.show()

## 4. Convert to PyTorch tensors

PyTorch 3D models (e.g. `torch.nn.Conv3d`) expect inputs laid out as `(channels, depth, height, width)`.

- The CT Volume has no channel dimension, but we want a pytorch tensor with a single channel, as is the standard format for grayscale images, so we add a leading axis: `(1, D, H, W)`.
- The mask Volume has one channel per segment in the last axis, so we move that axis to the front: `(K, D, H, W)`. Note that if you had used `combine_segments=True` above, the mask Volume would have no channels, and we would instead treat it just like we treat the CT volume.

In [ ]:
image = torch.from_numpy(ct_volume.array).unsqueeze(0)
target = torch.from_numpy(masks).permute(3, 0, 1, 2)

print("Image:", tuple(image.shape), image.dtype)
print("Masks:", tuple(target.shape), target.dtype)

## 5. Put it together in a `Dataset`

Finally, we combine the steps above into a function and use it in a minimal `torch.utils.data.Dataset` that loads
one CT/SEG pair per item. We also scale the HU values to the range [0, 1] using the same window as above, which is
a simple and common choice of intensity normalization for CT.

Since each CT has a different number of slices, we use `batch_size=1`. In a real training pipeline you would
typically add resampling, cropping or patch extraction here (for example using
[MONAI](https://monai.io/) transforms) so that cases can be batched together. For larger datasets, you may also
want to save the tensors to disk once (e.g. with `torch.save`) rather than parsing the DICOM files every epoch.

In [ ]:
def load_case(ct_dir: Path, seg_file: Path) -> tuple[torch.Tensor, torch.Tensor]:
    """Load a CT series and its SEG as (1, D, H, W) and (K, D, H, W) tensors."""
    ct_volume = hd.get_volume_from_series(
        [pydicom.dcmread(f) for f in sorted(ct_dir.glob("*.dcm"))],
        dtype=np.float32,
    )
    seg = hd.seg.segread(seg_file)
    mask_volume = seg.get_volume(combine_segments=False).match_geometry(ct_volume, tol=1e-4)

    image = torch.from_numpy(ct_volume.array).unsqueeze(0)
    masks = torch.from_numpy(mask_volume.array).permute(3, 0, 1, 2)
    return image, masks


class CTSegDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, download_dir: Path):
        self.pairs = pairs.reset_index(drop=True)
        self.download_dir = download_dir

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        row = self.pairs.iloc[i]
        ct_dir = self.download_dir / row.ct_series_uid
        seg_file = next((self.download_dir / row.seg_series_uid).glob("*.dcm"))
        image, masks = load_case(ct_dir, seg_file)
        image = (image.clamp(-200, 300) + 200) / 500  # window and scale to [0, 1]
        return image, masks

In [ ]:
dataset = CTSegDataset(pairs, download_dir)
loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

for image, masks in loader:
    print("Image batch:", tuple(image.shape), " Mask batch:", tuple(masks.shape))

Note that this assumes every SEG in the dataset contains the same segments in the same order (here, kidney then
mass), which is true for C4KC-KiTS. When combining data from different collections, check the segment
descriptions and reorder the channels so that each channel always means the same structure.

## Next steps

- Select a larger cohort by changing the `LIMIT` in the query, or explore other collections with segmentations
  using `seg_index`.
- If you will train and evaluate a model, split the data by patient (not by series) so that no patient
  appears in both the training and test sets.
- See the [highdicom documentation](https://highdicom.readthedocs.io/en/latest/volume.html) for more on working
  with volumes, including resampling and reorienting them.

## Support

For questions, use the [IDC user forum](https://discourse.canceridc.dev) or open an issue in
[IDC-Tutorials](https://github.com/ImagingDataCommons/IDC-Tutorials/issues).

## Acknowledgments

Imaging Data Commons has been funded in whole or in part with Federal funds from the National Cancer Institute,
National Institutes of Health, under Task Order No. HHSN26110071 under Contract No. HHSN261201500003I.

If you use IDC in research, cite Fedorov et al., *National Cancer Institute Imaging Data Commons: Toward
Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence*, RadioGraphics (2023),
[doi:10.1148/rg.230180](https://doi.org/10.1148/rg.230180).

The C4KC-KiTS data are provided under CC BY 3.0. Cite Heller et al., *Data from C4KC-KiTS*, The Cancer Imaging
Archive (2019), [doi:10.7937/TCIA.2019.IX49E8NX](https://doi.org/10.7937/TCIA.2019.IX49E8NX), and follow the
[collection's full citation instructions](https://www.cancerimagingarchive.net/collection/c4kc-kits/).